# Keep your notebook's files in Mountable

[Mountable](https://mountable.io/?utm_source=data-science-and-ml-workspaces&utm_medium=notebook&utm_content=nb-01-quickstart) is a persistent filesystem for notebooks and AI agents.
Files you save there stay when the kernel restarts, the session times out or you move to another notebook service.

This notebook saves a dataset, a feature cache and training checkpoints to one Mountable folder.
Run it, restart the kernel, and run it again: it picks up the files it already made instead of starting over.
It works the same in Google Colab, Kaggle, Deepnote, Hugging Face Spaces and Jupyter on your own machine.

**Before you start**

1. In the [Mountable console](https://mountable.io/app?utm_source=data-science-and-ml-workspaces&utm_medium=notebook&utm_content=nb-01-quickstart), create a filesystem and copy its ID.
2. Create an API key in **API keys**.
3. Add both as secrets named `MOUNTABLE_API_KEY` and `MOUNTABLE_FS_ID`:
   Colab **Secrets** (the key icon), Kaggle **Add-ons → Secrets**, Deepnote or Hugging Face environment variables,
   or plain environment variables elsewhere.

In [ ]:
%pip install -q mountable-sdk

## Connect

In [ ]:
import os

from mountable_sdk import Mountable, MountableError


def secret(name):
    """Environment variables, then Colab Secrets, then Kaggle Secrets."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata

        return userdata.get(name)
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except ImportError:
        return None


FS_ID = secret("MOUNTABLE_FS_ID")
if not FS_ID:
    raise ValueError("Add your filesystem ID as the secret MOUNTABLE_FS_ID")
client = Mountable(api_key=secret("MOUNTABLE_API_KEY"))
fs = client.filesystem(FS_ID, mode="rw")


def read_or_none(path):
    """The file's bytes, or None if it does not exist yet."""
    try:
        return fs.read(path).data
    except MountableError as e:
        if e.code == "not_found":
            return None
        raise

## Save the dataset once

The first run makes a small dataset and saves it. Later runs read it back from Mountable.
Swap in your own download here: it then happens once, not after every restart.

In [ ]:
import csv
import io
import random

DATASET = "datasets/houses.csv"

raw = read_or_none(DATASET)
if raw is None:
    rng = random.Random(42)
    rows = [
        (size, round(3 * size + 50 + rng.gauss(0, 10), 2)) for size in range(20, 220)
    ]
    out = io.StringIO()
    csv.writer(out).writerows([("size_m2", "price_k")] + rows)
    raw = out.getvalue().encode()
    fs.write(DATASET, raw)
    print(f"Saved {DATASET} ({len(rows)} rows) to Mountable")
else:
    print(f"Read {DATASET} from Mountable")

rows = list(csv.DictReader(io.StringIO(raw.decode())))

## Cache features

Feature engineering can take a long time. Keep the result in Mountable and compute it only when the cache is missing.

In [ ]:
import json
import statistics

FEATURES = "cache/features.json"

cached = read_or_none(FEATURES)
if cached is None:
    sizes = [float(r["size_m2"]) for r in rows]
    mean, stdev = statistics.mean(sizes), statistics.stdev(sizes)
    features = {
        "mean": mean,
        "stdev": stdev,
        "x": [(s - mean) / stdev for s in sizes],
        "y": [float(r["price_k"]) for r in rows],
    }
    fs.write(FEATURES, json.dumps(features))
    print(f"Computed features and cached them in {FEATURES}")
else:
    features = json.loads(cached)
    print(f"Loaded cached features from {FEATURES}")

## Train with checkpoints

The training loop saves a checkpoint after every epoch.
If the notebook stops halfway, the next run resumes from the last checkpoint.

In [ ]:
CHECKPOINT = "checkpoints/latest.json"
EPOCHS = 30

saved = read_or_none(CHECKPOINT)
state = json.loads(saved) if saved else {"epoch": 0, "w": 0.0, "b": 0.0}
print(f"Starting at epoch {state['epoch']} of {EPOCHS}")

x, y = features["x"], features["y"]
for epoch in range(state["epoch"] + 1, EPOCHS + 1):
    errors = [state["w"] * xi + state["b"] - yi for xi, yi in zip(x, y, strict=True)]
    state["w"] -= (
        0.1 * 2 * statistics.mean(e * xi for e, xi in zip(errors, x, strict=True))
    )
    state["b"] -= 0.1 * 2 * statistics.mean(errors)
    state["epoch"] = epoch
    state["loss"] = statistics.mean(e * e for e in errors)
    fs.write(CHECKPOINT, json.dumps(state))

print(f"Epoch {state['epoch']}: loss {state.get('loss', 0):.1f}")

## Mount the folder (where FUSE is available)

On a machine with FUSE (`/dev/fuse`), such as your own Linux box or GPU server, you can also mount the same folder.
Then any library reads and writes it with ordinary file paths.
The notebook creates a one-time mount ticket, so the mount process never sees your API key.

In [ ]:
import subprocess
import sys
import sysconfig

MOUNT = os.path.expanduser("~/mountable")

if os.path.exists("/dev/fuse"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mountable"], check=True
    )
    cli = os.path.join(sysconfig.get_path("scripts"), "mountable")
    os.makedirs(MOUNT, exist_ok=True)
    ticket = client.create_mount_ticket(FS_ID, mode="rw")
    mount = subprocess.Popen(
        [cli, "mount", "--ticket-stdin", "--json", MOUNT],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        text=True,
    )
    mount.stdin.write(ticket.ticket + "\n")
    mount.stdin.close()
    for line in mount.stdout:
        event = json.loads(line)
        if event["event"] != "mounted":
            raise RuntimeError(f"mount ended: {event}")
        break
    print(sorted(os.listdir(MOUNT)))
    with open(os.path.join(MOUNT, CHECKPOINT)) as f:
        print(json.load(f)["epoch"])
    # When you are done: subprocess.run([cli, "unmount", MOUNT])
else:
    print("No /dev/fuse here; the cells above work without it.")

## Next steps

- Open the same filesystem from another notebook with `mode="ro"` to read results without being able to change them.
- [Files API and SDKs](https://mountable.io/docs/sdks?utm_source=data-science-and-ml-workspaces&utm_medium=notebook&utm_content=nb-01-quickstart): conditional writes, listings and following changes.
- [Webhooks](https://mountable.io/docs/webhooks?utm_source=data-science-and-ml-workspaces&utm_medium=notebook&utm_content=nb-01-quickstart): start a downstream job when a new checkpoint lands.